# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hapepaAhmed/my-capstone-project/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip install -q huggingface_hub

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import userdata
from huggingface_hub import hf_hub_download

In [4]:
HF_TOKEN = userdata.get("HF_TOKEN")

In [5]:
parquet_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

In [6]:
df = pd.read_parquet(parquet_path)

print(df.shape)
df.head()

(9841378, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Answer:
  ## My Rule

I will prioritize content pages that receive many search impressions but have a relatively low average search position. These pages already have search visibility, so improving their ranking could increase clicks and organic traffic.

The baseline score will be based on:
- Higher Google Search impressions (`gsc_impressions`)
- Lower average search position (`gsc_avg_position`)

Pages with high impressions and lower rankings will receive higher priority scores.

## Reason Codes

The rule can output one of the following reason codes:

- **HIGH_IMPRESSIONS_LOW_RANK** – The page has many impressions but is not ranking highly, making it a good optimization candidate.
- **LOW_IMPRESSIONS** – The page has limited search visibility, so it is not currently a high priority.
- **GOOD_POSITION** – The page already ranks well, so immediate optimization is less important.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# Make a copy
baseline = df.copy()

# Normalize the signals (0-1 scale)
baseline["impressions_norm"] = (
    baseline["gsc_impressions"] - baseline["gsc_impressions"].min()
) / (
    baseline["gsc_impressions"].max() - baseline["gsc_impressions"].min()
)

baseline["position_norm"] = (
    baseline["gsc_avg_position"] - baseline["gsc_avg_position"].min()
) / (
    baseline["gsc_avg_position"].max() - baseline["gsc_avg_position"].min()
)

# Higher impressions + worse position = higher priority
baseline["baseline_score"] = (
    baseline["impressions_norm"] * 0.6 +
    baseline["position_norm"] * 0.4
)

# Reason code
baseline["reason_code"] = np.where(
    (baseline["gsc_impressions"] >= baseline["gsc_impressions"].median()) &
    (baseline["gsc_avg_position"] > baseline["gsc_avg_position"].median()),
    "HIGH_IMPRESSIONS_LOW_RANK",
    "LOW_PRIORITY"
)

# Action label
baseline["action"] = np.where(
    baseline["reason_code"] == "HIGH_IMPRESSIONS_LOW_RANK",
    "Optimize Content",
    "Monitor"
)

# Rank pages
baseline = baseline.sort_values(
    by="baseline_score",
    ascending=False
)

baseline.head(10)



,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,impressions_norm,position_norm,baseline_score,reason_code,action
9655081,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,True,True,True,False,40084,1,3341,...,0.0,0.0,0.0,0.0,0.0,1.000000,0.000167,0.600067,LOW_PRIORITY,Monitor
8054586,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,True,True,True,True,39305,252,86373,...,0.0,0.0,0.0,0.0,26.0,0.980566,0.004413,0.590105,LOW_PRIORITY,Monitor
103612,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,True,False,True,None,39003,2,107840,...,NaN,NaN,NaN,NaN,NaN,0.973032,0.005552,0.586040,LOW_PRIORITY,Monitor
9179913,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,True,True,True,True,38436,271,84405,...,0.0,0.0,0.0,0.0,16.0,0.958886,0.004410,0.577096,LOW_PRIORITY,Monitor
103661,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,True,False,True,None,37368,0,321886,...,NaN,NaN,NaN,NaN,NaN,0.932242,0.017297,0.566264,HIGH_IMPRESSIONS_LOW_RANK,Optimize Content
8972299,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,True,True,True,True,35404,225,77478,...,0.0,0.0,0.0,0.0,13.0,0.883245,0.004394,0.531705,LOW_PRIORITY,Monitor
8640840,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,True,True,True,True,34817,223,75948,...,0.0,0.0,0.0,0.0,13.0,0.868601,0.004380,0.522913,LOW_PRIORITY,Monitor
9767993,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,True,True,True,True,34606,235,77604,...,0.0,0.0,0.0,0.0,23.0,0.863337,0.004503,0.519803,LOW_PRIORITY,Monitor
7406798,2026-03-24,client_e547b89c05043229,content_eadb33b5df496f4a,True,True,True,True,33571,215,77517,...,0.0,0.0,0.0,0.0,14.0,0.837516,0.004637,0.504364,LOW_PRIORITY,Monitor
8882581,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,True,True,True,False,33383,0,6059,...,0.0,0.0,0.0,0.0,0.0,0.832826,0.000364,0.499841,LOW_PRIORITY,Monitor


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# Select the top 20 ranked pages
top20 = baseline.head(20)

top20[[
    "content_hash_id",
    "baseline_score",
    "action",
    "reason_code"
]]


,content_hash_id,baseline_score,action,reason_code
9655081,content_44f34c0a90047651,0.600067,Monitor,LOW_PRIORITY
8054586,content_eadb33b5df496f4a,0.590105,Monitor,LOW_PRIORITY
103612,content_34a70fea29d15f24,0.586040,Monitor,LOW_PRIORITY
9179913,content_eadb33b5df496f4a,0.577096,Monitor,LOW_PRIORITY
103661,content_945d6ff91386c817,0.566264,Optimize Content,HIGH_IMPRESSIONS_LOW_RANK
8972299,content_eadb33b5df496f4a,0.531705,Monitor,LOW_PRIORITY
8640840,content_eadb33b5df496f4a,0.522913,Monitor,LOW_PRIORITY
9767993,content_eadb33b5df496f4a,0.519803,Monitor,LOW_PRIORITY
7406798,content_eadb33b5df496f4a,0.504364,Monitor,LOW_PRIORITY
8882581,content_fec55986a1868d62,0.499841,Monitor,LOW_PRIORITY


In [9]:
top20_review = top20[[
    "content_hash_id",
    "baseline_score",
    "action",
    "reason_code"
]].copy()

top20_review["confidence_note"] = (
    "Medium - based on impressions and average position only."
)

top20_review["what_would_make_it_wrong"] = (
    "Missing context such as content quality, seasonality, or recent ranking changes."
)

top20_review

,content_hash_id,baseline_score,action,reason_code,confidence_note,what_would_make_it_wrong
9655081,content_44f34c0a90047651,0.600067,Monitor,LOW_PRIORITY,Medium - based on impressions and average posi...,"Missing context such as content quality, seaso..."
8054586,content_eadb33b5df496f4a,0.590105,Monitor,LOW_PRIORITY,Medium - based on impressions and average posi...,"Missing context such as content quality, seaso..."
103612,content_34a70fea29d15f24,0.586040,Monitor,LOW_PRIORITY,Medium - based on impressions and average posi...,"Missing context such as content quality, seaso..."
9179913,content_eadb33b5df496f4a,0.577096,Monitor,LOW_PRIORITY,Medium - based on impressions and average posi...,"Missing context such as content quality, seaso..."
103661,content_945d6ff91386c817,0.566264,Optimize Content,HIGH_IMPRESSIONS_LOW_RANK,Medium - based on impressions and average posi...,"Missing context such as content quality, seaso..."
8972299,content_eadb33b5df496f4a,0.531705,Monitor,LOW_PRIORITY,Medium - based on impressions and average posi...,"Missing context such as content quality, seaso..."
8640840,content_eadb33b5df496f4a,0.522913,Monitor,LOW_PRIORITY,Medium - based on impressions and average posi...,"Missing context such as content quality, seaso..."
9767993,content_eadb33b5df496f4a,0.519803,Monitor,LOW_PRIORITY,Medium - based on impressions and average posi...,"Missing context such as content quality, seaso..."
7406798,content_eadb33b5df496f4a,0.504364,Monitor,LOW_PRIORITY,Medium - based on impressions and average posi...,"Missing context such as content quality, seaso..."
8882581,content_fec55986a1868d62,0.499841,Monitor,LOW_PRIORITY,Medium - based on impressions and average posi...,"Missing context such as content quality, seaso..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# Features used in the baseline rule
used_features = [
    "gsc_impressions",
    "gsc_avg_position"
]

print("Features used in the baseline:")
for feature in used_features:
    print("-", feature)

Features used in the baseline:
- gsc_impressions
- gsc_avg_position


In [11]:
# Look for columns that could indicate future windows
future_columns = [
    col for col in df.columns
    if "future" in col.lower()
    or "next" in col.lower()
    or "label" in col.lower()
]

print("Potential future/label columns:")
print(future_columns if future_columns else "None found")

Potential future/label columns:
None found


In [12]:
# Look for product-related columns
product_columns = [
    col for col in df.columns
    if "product" in col.lower() or "flag" in col.lower()
]

print("Potential product flag columns:")
print(product_columns if product_columns else "None found")

Potential product flag columns:
None found


In [13]:
print("Columns used to calculate baseline_score:")
print(used_features)

assert set(used_features) == {"gsc_impressions", "gsc_avg_position"}

print("Leakage check passed: baseline score uses only current ranking signals.")

Columns used to calculate baseline_score:
['gsc_impressions', 'gsc_avg_position']
Leakage check passed: baseline score uses only current ranking signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.